# Create Stuff Document Chain

In [1]:
from langchain_community.document_loaders import WebBaseLoader

/tmp/ipykernel_27483/967698044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/home/ajayp/anaconda3/envs/langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
URL1 = "https://techcrunch.com/2024/02/27/microsoft-made-a-16-million-investment-in-mistral-ai/"
URL2 = "https://techcrunch.com/2024/03/28/ai21-labs-new-text-generating-ai-model-is-more-efficient-than-most/"

loader = WebBaseLoader([URL1, URL2])
data = loader.load()

print(len(data))  # Expect: 2

2


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [("system", "What models are launched by Mistral and AI21 Labs:\n\n{context}")]
)

In [6]:

import os

from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b"
)
chain = create_stuff_documents_chain(llm, prompt)

In [8]:
result = chain.invoke({"context": data})
print(result)

**Mistral AI**

- **Model:** **Mistral Large** – the company’s flagship large‑language model introduced in February 2024. It is a closed‑source model aimed at competing with GPT‑4, Claude 2, etc., and is accessible via Mistral’s own API (and later through Azure’s model catalog).

**AI21 Labs**

- **Model:** **Jamba** – the new generative text model announced in March 2024. Jamba can process up to 140 000 tokens (≈105 000 words) on a single GPU with ≥80 GB memory, uses a hybrid transformer + state‑space‑model (SSM) architecture (leveraging the “Mamba” SSM), and is released under the Apache 2.0 license (research‑only release).


# Create Retrival Chain

In [ ]:
# Imports
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_community.vectorstores import FAISS

# URLs to load
URL1 = "https://techcrunch.com/2024/03/04/anthropic-claims-its-new-models-beat-gpt-4/"
URL2 = "https://techcrunch.com/2024/03/28/ai21-labs-new-text-generating-ai-model-is-more-efficient-than-most/"

# 1) Load documents from the web
loader = WebBaseLoader([URL1, URL2])
data = loader.load()

# 2) Split documents into chunks (tune chunk_size/overlap for your documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(data)

# 3) Create embeddings for the chunks
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4) Build an in-memory FAISS vector store and get a retriever
# Note: FAISS is in-memory by default (non-persistent).
vector = FAISS.from_documents(chunks, embeddings)
retriever = vector.as_retriever()

# 5) Define a prompt template that forces the LLM to answer only from context
prompt_template = """Answer the question {input} based solely on the context below:

<context>
{context}
</context>

If you can't find an answer, say "I don't know."
"""
prompt = PromptTemplate.from_template(prompt_template)

# 6) Initialize the LLM with low temperature to reduce creativity/hallucination
llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="gpt-oss-120b",
    temperature=0.0)

# 7) Create the documents combiner chain and the retrieval chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, combine_docs_chain)

# 8) Invoke the chain with a question that restricts answers to Anthropic and Meta only
result = chain.invoke({
    "input": "List the models and their token size of models only from Anthropic and Meta"
})

# 9) Print the returned answer
print(result["answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 720.75it/s]


I don't know.
